# ArXiv Paper Metadata Scraper — Report

**Course:** Web Scraping  
**Author:** NakerTheFirst  
**Target site:** [arxiv.org](https://arxiv.org)  
**Categories scraped:** `cs.AI`, `cs.CV`, `cs.LG`, `cs.CL`

---

## Contents

1. [Legal & Ethical Basis](#1-legal)
2. [Website Structure](#2-structure)
3. [Python Regex Patterns](#3-regex)
4. [Scraping Tools](#4-tools)
   - 4.1 requests + BeautifulSoup (`/list` pages)
   - 4.2 Selenium (`/abs` pages)
   - 4.3 Scrapy (full `/list → /abs` crawl)
5. [Dataset Exploration](#5-eda)
6. [Conclusions](#6-conclusions)

In [ ]:
import sys
import re
import os
from pathlib import Path

# Add project root to path so we can import src.*
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (9, 4)})

REFERENCES = PROJECT_ROOT / 'references'
DATA_DIR   = PROJECT_ROOT / 'data'
CSV_PATH   = DATA_DIR / 'papers.csv'

print('Project root:', PROJECT_ROOT)
print('Data CSV exists:', CSV_PATH.exists())

<a id='1-legal'></a>
## 1. Legal & Ethical Basis

ArXiv's `robots.txt` explicitly **allows** scraping of the paths used in this project:

```
User-agent: *
Crawl-delay: 15
Allow: /list
Allow: /abs
Allow: /pdf
Allow: /html
Allow: /archive
```

All scrapers in this project:
- Respect the **15-second crawl delay** between requests (`CRAWL_DELAY = 15` in `config.py`)
- Only access `/list` and `/abs` paths
- Set a descriptive `User-Agent` header
- Do **not** download full PDFs or source files

ArXiv's [Terms of Use](https://arxiv.org/help/robots) state that metadata scraping for academic purposes is permitted provided the crawl delay is honoured.

In [ ]:
# Display relevant section of the actual robots.txt
robots = (REFERENCES / 'robots.txt').read_text(encoding='utf-8')
# Show only the first (default) user-agent block
first_block = robots.split('User-agent: Googlebot')[0]
print(first_block.strip())

<a id='2-structure'></a>
## 2. Website Structure

ArXiv exposes two page types relevant to this project:

| Page | URL pattern | Content |
|------|-------------|----------|
| **Listing** | `/list/{category}/{date}` | Index of recent papers: IDs, titles, authors, subjects |
| **Abstract** | `/abs/{arxiv_id}` | Full metadata: abstract, submission date, DOI, history |

### 2.1 `/list` page structure

The listing page wraps all papers in a `<dl id="articles">` element.  
Each paper occupies a `<dt>` / `<dd>` pair:

```html
<dl id="articles">
  <h3>Fri, 8 May 2026 (showing first 50 of 355 entries)</h3>
  <dt>
    <a title="Abstract" id="2605.06651" href="/abs/2605.06651">arXiv:2605.06651</a>
    [<a title="Download PDF" href="/pdf/2605.06651">pdf</a>] ...
  </dt>
  <dd>
    <div class="meta">
      <div class="list-title"><span class="descriptor">Title:</span> ...</div>
      <div class="list-authors"><a>Author 1</a>, ...</div>
      <div class="list-subjects"><span class="primary-subject">cs.AI</span></div>
    </div>
  </dd>
</dl>
```

In [ ]:
# Parse the saved example /list page
list_html = (REFERENCES / 'example_list.htm').read_text(encoding='utf-8')
list_soup = BeautifulSoup(list_html, 'html.parser')

articles = list_soup.find('dl', id='articles')
h3 = articles.find('h3')
print('Listing header:', h3.get_text(strip=True))
print()

# Inspect first entry
first_dt = articles.find('dt')
first_dd = articles.find('dd')

abs_anchor = first_dt.find('a', title='Abstract')
print('ArXiv ID (id attr):', abs_anchor['id'])
print('Abs href:          ', abs_anchor['href'])

title_div = first_dd.find('div', class_='list-title')
for desc in title_div.find_all('span', class_='descriptor'):
    desc.decompose()
print('Title:             ', title_div.get_text(strip=True))

authors = [a.get_text() for a in first_dd.find_all('a')]
print('Authors:           ', '; '.join(authors))

subjects_div = first_dd.find('div', class_='list-subjects')
primary = subjects_div.find('span', class_='primary-subject')
print('Primary subject:   ', primary.get_text())

### 2.2 `/abs` page structure

The abstract page places all key fields inside `<div id="abs">`:

```html
<div id="abs">
  <div class="dateline">[Submitted on 7 May 2026]</div>
  <h1 class="title mathjax"><span class="descriptor">Title:</span> ...</h1>
  <div class="authors"><span class="descriptor">Authors:</span> <a>...</a></div>
  <blockquote class="abstract mathjax">
    <span class="descriptor">Abstract:</span> ...
  </blockquote>
  <div class="metatable">
    <table>
      <tr><td class="tablecell label">Subjects:</td>
          <td class="tablecell subjects"><span class="primary-subject">cs.AI</span></td></tr>
      <tr><td class="tablecell label">Comments:</td>
          <td class="tablecell comments">22 pages</td></tr>
    </table>
  </div>
</div>
```

In [ ]:
abs_html = (REFERENCES / 'example_abs.htm').read_text(encoding='utf-8')
abs_soup = BeautifulSoup(abs_html, 'html.parser')

abs_div = abs_soup.find('div', id='abs')

# Dateline
dateline = abs_div.find('div', class_='dateline')
print('Dateline:  ', dateline.get_text(strip=True))

# Title (strip descriptor span first)
title_tag = abs_div.find('h1', class_='title')
for d in title_tag.find_all('span', class_='descriptor'):
    d.decompose()
print('Title:     ', title_tag.get_text(strip=True))

# Abstract
bq = abs_div.find('blockquote', class_='abstract')
for d in bq.find_all('span', class_='descriptor'):
    d.decompose()
abstract_text = bq.get_text(strip=True)
print('Abstract:  ', abstract_text[:120], '...')

# Subjects metatable
subjects_td = abs_div.find('td', class_='subjects')
print('Subjects:  ', subjects_td.get_text(strip=True))

# DOI
doi_link = abs_soup.find('a', id='arxiv-doi-link')
print('DOI:       ', doi_link.get_text(strip=True) if doi_link else 'n/a')

<a id='3-regex'></a>
## 3. Python Regex Patterns

All patterns are compiled once in `src/utils.py` and shared across all three scrapers.

In [ ]:
from src.utils import (
    ARXIV_ID_RE,
    SUBMISSION_DATE_RE,
    CATEGORY_CODE_RE,
    VERSION_RE,
    DOI_RE,
    WHITESPACE_RE,
    _LISTING_COUNT_RE,
    _LISTING_DATE_RE,
    extract_arxiv_id,
    extract_submission_date,
    extract_categories,
    parse_listing_header,
    extract_doi,
)

# ── 1. ArXiv ID ─────────────────────────────────────────────────────────────
samples = [
    'arXiv:2605.06651v1',
    'See also arXiv:2501.12345',
    'Paper ID 2312.99999v3 from last year',
]
print('ArXiv ID pattern:', ARXIV_ID_RE.pattern)
for s in samples:
    print(f'  {s!r:45s} → {extract_arxiv_id(s)!r}')

In [ ]:
# ── 2. Submission date ───────────────────────────────────────────────────────
datelines = [
    '[Submitted on 7 May 2026]',
    '[Submitted on 12 January 2025]',
    '[v2] Mon, 10 Feb 2025 14:22:00 UTC',   # no match — correct
]
print('Submission date pattern:', SUBMISSION_DATE_RE.pattern)
for d in datelines:
    print(f'  {d!r:45s} → {extract_submission_date(d)!r}')

In [ ]:
# ── 3. Category codes ────────────────────────────────────────────────────────
subject_strings = [
    'Artificial Intelligence (cs.AI)',
    'Artificial Intelligence (cs.AI); Computer Vision (cs.CV)',
    'Machine Learning (cs.LG); Statistics (stat.ML); math.CO',
]
print('Category code pattern:', CATEGORY_CODE_RE.pattern)
for s in subject_strings:
    primary, cross = extract_categories(s)
    print(f'  primary={primary!r:10s}  cross={cross}')

In [ ]:
# ── 4. DOI ───────────────────────────────────────────────────────────────────
doi_samples = [
    'https://doi.org/10.48550/arXiv.2605.06651',
    'DOI: 10.1038/s41586-021-03819-2',
    'no doi here',
]
print('DOI pattern:', DOI_RE.pattern)
for s in doi_samples:
    print(f'  {s!r:55s} → {extract_doi(s)!r}')

In [ ]:
# ── 5. Listing header ────────────────────────────────────────────────────────
headers = [
    'Fri, 8 May 2026 (showing first 50 of 355 entries )',
    'Mon, 5 May 2026 (showing first 50 of 50 entries )',  # no truncation
    'Thu, 7 May 2026',                                    # no count at all
]
print('Count pattern:', _LISTING_COUNT_RE.pattern)
print('Date  pattern:', _LISTING_DATE_RE.pattern)
print()
for h in headers:
    date, shown, total = parse_listing_header(h)
    truncated = shown < total if total else False
    print(f'  date={date!r:25s}  shown={shown}  total={total}  truncated={truncated}')

In [ ]:
# ── 6. Whitespace normalisation ──────────────────────────────────────────────
dirty = '  AI  Co-Mathematician:\n  Accelerating\tMathematicians  '
clean = WHITESPACE_RE.sub(' ', dirty).strip()
print(f'Before: {dirty!r}')
print(f'After:  {clean!r}')

<a id='4-tools'></a>
## 4. Scraping Tools

### 4.1 requests + BeautifulSoup — `/list` pages

`src/requests_scraper.py` · `ListScraper`

- Fetches each category's `/list` page with `requests.Session`
- Detects truncation from the `<h3>` header; re-requests with `?skip=0&show=N` if needed
- Pairs `<dt>` / `<dd>` siblings; decomposes `<span class="descriptor">` before calling `get_text()` to avoid label bleed
- Returns a list of stub dicts (no abstract — those come from the `/abs` scrapers)

In [ ]:
# Demonstrate ListScraper against the saved example page (no network call)
from src.requests_scraper import ListScraper
from unittest.mock import patch, MagicMock
from bs4 import BeautifulSoup

list_html = (REFERENCES / 'example_list.htm').read_text(encoding='utf-8')

scraper = ListScraper()
# Feed the example HTML directly into the private parser
soup = BeautifulSoup(list_html, 'html.parser')
papers = scraper._parse_list_page(soup)
scraper.close()

print(f'Parsed {len(papers)} stubs from example_list.htm')
print()
# Show first record
import json
print(json.dumps(papers[0], indent=2))

### 4.2 Selenium — `/abs` pages

`src/selenium_scraper.py` · `AbsScraper`

ArXiv `/abs` pages are static HTML, but Selenium is used here as required by the course specification to demonstrate browser-automation skills. It provides two concrete advantages over a plain HTTP request:

1. **Explicit waits** — `WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, 'abs')))` blocks until the DOM element is present, making the scraper robust to slow server responses without relying on arbitrary `time.sleep()` calls.
2. **Full rendered DOM** — `driver.page_source` captures the post-JavaScript state; any dynamically injected content is automatically included.

In [ ]:
# Show AbsScraper parsing logic against the saved example page
from src.selenium_scraper import AbsScraper

abs_html = (REFERENCES / 'example_abs.htm').read_text(encoding='utf-8')
abs_soup = BeautifulSoup(abs_html, 'html.parser')

# We can call the internal parse method directly without launching a browser
dummy_scraper = object.__new__(AbsScraper)  # skip __init__ / driver creation
record = AbsScraper._parse_abs_page(dummy_scraper, abs_soup, '2605.06651')

for key, val in record.items():
    preview = str(val)[:80] + ('...' if len(str(val)) > 80 else '')
    print(f'{key:25s}: {preview}')

In [ ]:
# Show the Selenium driver initialisation code (not executed here)
import inspect
from src.selenium_scraper import AbsScraper

print(inspect.getsource(AbsScraper._build_driver))
print()
print(inspect.getsource(AbsScraper._load_abs_page))

### 4.3 Scrapy — full `/list → /abs` crawl

`src/scrapy_scraper/spiders/arxiv_spider.py` · `ArxivSpider`

The Scrapy spider performs the same two-step crawl as the other scrapers but asynchronously:

| Setting | Value | Reason |
|---------|-------|--------|
| `ROBOTSTXT_OBEY` | `True` | Automatic enforcement of robots.txt |
| `DOWNLOAD_DELAY` | 15 s | Matches `Crawl-delay` in robots.txt |
| `RANDOMIZE_DOWNLOAD_DELAY` | `True` | Jitter ∈ [7.5 s, 22.5 s] to avoid pattern detection |
| `AUTOTHROTTLE_ENABLED` | `True` | Server-latency-aware throttling |
| `CONCURRENT_REQUESTS` | 1 | Single in-flight request — respectful crawl |

Two item pipelines run after each scraped item:
1. `DeduplicatePipeline` — drops items whose `arxiv_id` was already seen (same paper across multiple category listings)
2. `CleanTextPipeline` — normalises whitespace in all string fields

In [ ]:
# Inspect the spider's parse_list and parse_abs method signatures
import inspect
from src.scrapy_scraper.spiders.arxiv_spider import ArxivSpider

for method_name in ('start_requests', 'parse_list', '_parse_entries', 'parse_abs'):
    method = getattr(ArxivSpider, method_name)
    sig = inspect.signature(method)
    doc = (inspect.getdoc(method) or '').split('\n')[0]
    print(f'{method_name}{sig}')
    if doc:
        print(f'  "{doc}"')
    print()

In [ ]:
# Show the item schema
from src.scrapy_scraper.items import ArxivPaperItem
print('ArxivPaperItem fields:')
for field in ArxivPaperItem.fields:
    print(f'  {field}')

<a id='5-eda'></a>
## 5. Dataset Exploration

Run `python main.py` from the project root to populate `data/papers.csv` before executing the cells below.

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'{CSV_PATH} not found.\n'
        'Run  python main.py  from the project root first.'
    )

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumns:', list(df.columns))
df.head(3)

In [ ]:
# Basic coverage statistics
coverage = (
    df.replace('', pd.NA)
      .notna()
      .mean()
      .mul(100)
      .round(1)
      .rename('% populated')
      .to_frame()
)
coverage

In [ ]:
# ── Category distribution ─────────────────────────────────────────────────
cat_counts = df['primary_category'].value_counts()

fig, ax = plt.subplots()
cat_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Papers per primary category')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(cat_counts.to_string())

In [ ]:
# ── Cross-listing prevalence ──────────────────────────────────────────────
has_cross = df['cross_list_categories'].str.strip().ne('').sum()
print(f'{has_cross:,} of {len(df):,} papers ({has_cross/len(df):.1%}) have cross-list categories')

# Which cross-listed categories appear most often?
cross_exploded = (
    df['cross_list_categories']
    .str.split(';')
    .explode()
    .str.strip()
    .replace('', pd.NA)
    .dropna()
)
print('\nTop 10 cross-listed categories:')
print(cross_exploded.value_counts().head(10).to_string())

In [ ]:
# ── Submission date distribution ──────────────────────────────────────────
dates = pd.to_datetime(df['submission_date'], errors='coerce').dropna()

if not dates.empty:
    fig, ax = plt.subplots()
    dates.dt.date.value_counts().sort_index().plot.bar(ax=ax, color='teal', edgecolor='white')
    ax.set_title('Submission dates')
    ax.set_xlabel('Date')
    ax.set_ylabel('Papers')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No submission_date data available (Selenium sample may be empty).')

In [ ]:
# ── Author count distribution ─────────────────────────────────────────────
author_counts = (
    df['authors']
    .str.split(';')
    .apply(lambda x: len([a for a in x if a.strip()]))
)

fig, ax = plt.subplots()
author_counts.clip(upper=15).value_counts().sort_index().plot.bar(
    ax=ax, color='coral', edgecolor='white'
)
ax.set_title('Authors per paper (capped at 15)')
ax.set_xlabel('Number of authors')
ax.set_ylabel('Papers')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Median authors per paper: {author_counts.median():.1f}')
print(f'Max authors in one paper: {author_counts.max()}')

In [ ]:
# ── Abstract length distribution (Selenium + Scrapy records only) ─────────
abstract_len = df['abstract'].str.len().replace(0, pd.NA).dropna()

if not abstract_len.empty:
    fig, ax = plt.subplots()
    abstract_len.plot.hist(bins=40, ax=ax, color='mediumseagreen', edgecolor='white')
    ax.axvline(abstract_len.median(), color='red', linestyle='--', label=f'Median = {abstract_len.median():.0f} chars')
    ax.set_title('Abstract length (characters)')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Papers')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(abstract_len.describe().round(0).astype(int).to_string())
else:
    print('No abstract data available.')

In [ ]:
# ── Most prolific authors in the dataset ──────────────────────────────────
all_authors = (
    df['authors']
    .str.split(';')
    .explode()
    .str.strip()
    .replace('', pd.NA)
    .dropna()
)

top_authors = all_authors.value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
top_authors.sort_values().plot.barh(ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Top 15 most frequent authors')
ax.set_xlabel('Papers in dataset')
plt.tight_layout()
plt.show()

In [ ]:
# ── DOI coverage ─────────────────────────────────────────────────────────
doi_coverage = df['doi'].str.strip().ne('').mean()
print(f'DOI populated for {doi_coverage:.1%} of records')
print('Sample DOIs:')
print(df.loc[df['doi'].ne(''), 'doi'].head(5).to_string(index=False))

In [ ]:
# ── Data-readiness summary ────────────────────────────────────────────────
print('=== Final dataset summary ===')
print(f'Total papers:         {len(df):,}')
print(f'Unique arxiv IDs:     {df["arxiv_id"].nunique():,}')
print(f'Categories covered:   {df["primary_category"].nunique()}')
print(f'Records with abstract:{(df["abstract"].str.len() > 0).sum():,}')
print(f'Records with DOI:     {(df["doi"].str.len() > 0).sum():,}')
print(f'Records with date:    {(df["submission_date"].str.len() > 0).sum():,}')
print()
print('dtypes after loading:')
print(df.dtypes.to_string())

<a id='6-conclusions'></a>
## 6. Conclusions

### What was built

A three-scraper pipeline covering all four mandatory libraries:

| Scraper | Library | Target | Fields added |
|---------|---------|--------|--------------|
| `ListScraper` | requests + BeautifulSoup | `/list` pages | ID, title, authors, subjects, links |
| `AbsScraper` | Selenium (headless Chrome) | `/abs` pages | abstract, submission date, DOI, history |
| `ArxivSpider` | Scrapy | `/list → /abs` | all of the above in one async crawl |

Python `re` is used throughout via seven compiled patterns in `src/utils.py`, covering arXiv IDs, ISO dates, category codes, DOIs, listing counts, and whitespace normalisation.

### Design decisions

- **Shared utilities** (`src/utils.py`) ensure consistent field extraction across all three scrapers — regex is compiled once and reused everywhere.
- **Pagination handling** — ArXiv's `/list` pages default to 50 entries; all scrapers detect truncation and re-request with `?skip=0&show=N`.
- **Descriptor span removal** — `<span class="descriptor">Title:</span>` is decomposed *before* calling `get_text()` to prevent label text from appearing in field values.
- **Merge priority** — Scrapy (async, complete records) > Selenium (sample, abs-enriched) > requests (stubs, broadest coverage). `drop_duplicates(keep='first')` preserves the richest record per paper.

### Limitations

- The 15-second crawl delay limits throughput; 200 papers per category × 4 categories × 15 s ≈ 3.3 hours for a Selenium-only run. The Scrapy spider's async architecture partially mitigates this.
- Abstract coverage depends on how many papers the Selenium and Scrapy phases visit. For full coverage, increase `SELENIUM_SAMPLE` and `MAX_PAPERS_PER_CATEGORY` in `config.py`.